## Leaky Integrate-and-Fire (LIF)
In previous neuroscience lectures, we've discussed action potentials and neurons. Today, we get to model these to simulate our own neuron with Python. The LIF model is extremely interdisciplinary, drawing on concepts from math, physics, neuroscience, and computing.

In our code, we will be using SI units throughout. This means, for example, if the value $100\text{km}$ is requested, you should write `100e3` rather than `100`.

Like usual, let's start with importing necessary module(s). This time, you'll be importing them. Remember to install them first! We want to
1. be able to efficiently manipulate numbers
2. graph data

In [ ]:
# TODO: install libraries we will be using
!pip install

In [ ]:
# TODO: import libraries
import

### Introduction
Before any coding, we need to understand the mechanics of the LIF model.

#### Modeling Voltage
Physics background:
- Potential: how difficult it is to move a charge from one point to another
- Voltage: the difference in potential between two points
- Current: change in charge per unit time
- Capacitor: stores charge

The heart of LIF lies in how we represent the neuron membrane as an RC (resistor-capacitor) circuit. We have a capacitor in parallel with a resistor; applied current splits current between these two.

Recall how there is a membrane potential in neurons: there is a difference in voltage between the two sides of the membrane. We call this $V_m$. Because the voltage is not a constant value, we define $V_m$ as a function of time $t$, i.e. $V_m(t)$, to model how voltage changes through time.

Below, declare variables `vth` and `vl` (more on these later) with values $-55\text{mV}$ and $-70\text{mV}$, respectively. Additionally, define `vreset` to have the same value as `vl`.

In [ ]:
# TODO: declare and initialize vth, vl, vreset with the provided values

#### Capacitance
Next, we model the membrane as a capacitor. The membrane stores, and thus integrates, charge through time. The ability to store charge is modified by capacitance $C$. The formula for capacitance is $C = \frac{Q}{V_m} \implies CV = Q$. Assuming capacitance is constant, we differentiate both sides with respect to time to get
$$
C_m\frac{dV_m}{dt} = \frac{dQ}{dt} = I_C(t)
$$
Remember that current is the change in charge per time period, exactly what $\frac{dQ}{dt}$ is.

#### Leak Channels
Other components we need to worry about are the leak channels. In order to maintain resting potential, neurons have channels that leak ions. This is where we get the term "leaky". These channels have total small conductance $G_L$ in order to not leak too many ions at once. Using Ohm's law, $V=IR=\frac{I}{G}$, we represent the total current as $GV$.

There exists a "leak potential" $V_L$, also known as the leak reversal potential. In order to model the potential across these leak channels, we use the equation $V_m(t) - V_L$. When $V_m(t) = V_L$, there is no leak current. When the potential is too high, there is positive current out of the cell. Likewise, when the potential is too little, there is negative current into the cell. The leak potential is not a function of time, rather a constant. The leak potential is equal to the resting membrane potential in our LIF model.

Using the Ohm's law, we get $I_L = G_L (V_m(t) - V_L)$.

In the LIF model, this is the only resistor in the membrane. Thus, $R_M = R_L \implies G_M = G_L$.

Below, define `rm` to be $100\text{M}\Omega$.

In [ ]:
# TODO: declare and initialize rm with the provided value

#### Modeling Current
Additionally, there would be multiple currents going either in or out of the membrane. Like with voltage, current $I$ varies with time, so we define $I$ as a model of $t$. For now, consider the applied current to be $I(t)$.

We use Kirchhoff's Junction Law in order to realize that the net current out is equal to the net current in. Thus, we can set up the following equation:
$$
I = G_L(V_m(t) - V_L) + C_m\frac{dV_m}{dt}
$$


#### Putting it all together
We introduce the time constant $\tau$, which is the time it takes to charge or discharge $1 - \frac{1}{e}$ of the capacitor's final / initial (depending on whether we're charging or discharging) voltage. The time constant $\tau_m = \frac{C_m}{G_m}$. We divide the equation by $G_m=G_L$ to get
$$
\frac{I}{G_m} = (V_m(t) - V_L) + \frac{C_m}{G_m}\frac{dV_m}{dt} \\
\frac{I}{G_m} = (V_m(t) - V_L) + \tau_m \frac{dV_m}{dt}
$$
Using $R = \frac{1}{G}$ and rearranging variables, we get the leaky integrate-and-fire equation:
$$
\tau_m\frac{dV_m}{dt} = -(V_m(t) - V_L) + RI(t)
$$

Below, define `tau` to be $20\text{ms}$.

In [ ]:
# TODO: declare and initialize tau with the provided value

### Firing
A notable portion that we're missing is neuron firing. Once the voltage of a neuron reaches a threshold voltage, it fires an action potential. During the time of the action potential, the neuron is unable to fire another action potential - this is the refractory period.

In our LIF model, we do not model the actual shape of the action potential. Instead, we plot the voltage as $+30\text{mV}$ for an instant before holding the voltage at the resting potential for the duration of the refractory period.

Below, define `ref` to be $3\text{ms}$.

In [ ]:
# TODO: declare and initialize ref with the value provided above

### More About Current
Current could either be constant or changing. We'll start by considering the former case.

#### Case 1: Constant Current
As time goes on, with a constant current, the capacitor wants to approach "steady state," when there is no change in voltage. We call this $V_\infty$. Because $\frac{dV_m}{dt} = 0 $ when $V_m(t) = V_\infty$, we get the following equation for steady state voltage:
$$
V_\infty = V_L + RI
$$
We plug back the steady state voltage into the LIF equation to get
$$
\tau_m \frac{dV_m}{dt} = V_\infty - V_m(t)
$$

We then solve the differential equation presented above with $V_\infty$ by separation of variables
$$
\frac{dV_m}{V_\infty - V_m(t)} = \frac{dt}{\tau_m} \\
\int\frac{dV_m}{V_\infty - V_m(t)} = \int\frac{dt}{\tau_m} \\
u = V_\infty - V_m(t) \implies du = -dV_m \\
-\int\frac{du}{u} = \int\frac{dt}{\tau_m} \\
-\ln u = \frac{t}{\tau_m} + C_0 \\
\ln u = -\frac{t}{\tau_m} - C_0 \\
V_\infty - V_m(t) = C_1e^{-\frac{t}{\tau_m}}
$$
In order to find constant of integration, we let $t = 0$. At $t=0$, we start at resting potential. Then,
$$
V_m(0) = V_\infty - C_1 \\
C_1 = V_\infty - V_m(0) \\
V_m(t) = V_\infty - (V_\infty - V_m(0))e^{-\frac{t}{\tau_m}} = V_\infty - (V_\infty - V_L)e^{-\frac{t}{\tau_m}} \\
V_m(t) = V_L + RI - (V_L + RI - V_L)e^{-\frac{t}{\tau_m}} \\
V_m(t) = V_L + RI - RIe^{-\frac{t}{\tau_m}} \\
\boxed{\therefore V_m(t) = V_L + RI(1 - e^{-\frac{t}{\tau_m}})}
$$

Consider a constant current $I$ to be $0.2\text{nA}$.

With these equations, let's simulate the membrane potential $V_m$ for $120\text{ms}$, making sure to model spikes as they happen. Note that we treat the $t$ in the exponential as the time since the last spike occurred.


In [ ]:
def constant_current():
    time = 120e-3
    dt = 0.1e-3

    # All the timesteps
    t = np.arange(0, time, dt)

    # TODO: number of samples is length of t
    samples =

    # TODO: initialize voltage as a numpy array of [samples] zeros
    #       set the initial voltage to be resting potential
    v =
    v[0] =

    # TODO: initialize current to be an array of [samples] .2nA
    I =

    # Time left for refractory period
    ref_left = 0

    # Time of charging starting
    charge_start = 0

    # We use samples - 1 because we update the next value; prevent out of bounds
    for step in range(samples - 1):
        # TODO: maintain neuron at reset potential
        if ref_left > 0:
            v[step + 1] =
            ref_left -= 1

            if ref_left == 0:
                charge_start = t[step + 1]

            continue

        # TODO: calculate time elapsed since last spike
        elapsed =

        # TODO: calculate the voltage at the next step
        v[step + 1] =

        # TODO: condition where we spike
        if :
            # TODO: set the next voltage to +30mV
            v[step + 1] =
            # TODO: how many "steps" is ref period given ref seconds, change dt?
            ref_left =


    # Plot voltage/current vs time
    fig, ax = plt.subplots(2, 1, sharex=True)

    ax[0].plot(t * 1e3, I * 1e9)
    ax[0].set_ylabel("Current (nA)")
    ax[0].set_title("Input Current")

    ax[1].plot(t * 1e3, v * 1e3)
    ax[1].set_ylabel("Voltage (mV)")
    ax[1].set_title("Membrane Voltage")

    ax[1].set_xlabel("Time (ms)")

    plt.tight_layout()
    plt.show()

constant_current()


#### Case 2: Changing Current
With changing current, we can no longer use the equation above. Instead, we use Forward Euler.

Recall the LIF equation:
$$
\tau_m\frac{dV_m}{dt} = -(V_m(t) - V_L) + RI(t)
$$

To solve this differential equation for $V_m(t)$, we use the Forward Euler method
$$
V_m(t+\Delta t) = V_m(t) + \frac{dV_m}{dt}\Delta t \\
V_m(t + \Delta t) = V_m(t) + \frac{-(V_m(t) - V_L) + RI(t)}{\tau_m}\Delta t
$$
Let's do a similar model below, but this time we model the current as a sine wave.


In [ ]:
def changing_current():
    time = 120e-3
    dt = 0.1e-3

    # All the timesteps
    t = np.arange(0, time, dt)

    # TODO: number of samples is length of t
    samples =

    # TODO: initialize voltage as a numpy array of [samples] zeros
    #       set the initial voltage to be resting potential
    v =
    v[0] =

    # TODO: current is sine wave with baseline .2nA, amplitude .1nA, frequency
    #       20Hz
    I =

    # Time left for refractory period
    ref_left = 0

    # We use samples - 1 because we update the next value; prevent out of bounds
    for step in range(samples - 1):
        # TODO: maintain neuron at reset potential
        if ref_left > 0:
            v[step + 1] =
            ref_left -= 1

            continue

        # TODO: calculate dV/dt
        dvdt =

        # TODO: calculate the next V with forward euler
        v[step + 1] =

        # TODO: condition where we spike
        if :
            # TODO: set the next voltage to +30mV
            v[step + 1] =
            # TODO: how many "steps" is ref period given ref seconds, change dt?
            ref_left =


    # Plot voltage/current vs time
    fig, ax = plt.subplots(2, 1, sharex=True)

    ax[0].plot(t * 1e3, I * 1e9)
    ax[0].set_ylabel("Current (nA)")
    ax[0].set_title("Input Current")

    ax[1].plot(t * 1e3, v * 1e3)
    ax[1].set_ylabel("Voltage (mV)")
    ax[1].set_title("Membrane Voltage")

    ax[1].set_xlabel("Time (ms)")

    plt.tight_layout()
    plt.show()

changing_current()


### f-I curve
Now, we go back to considering constant current.

The f-I curve describes the firing rate $f$ as a function of a current $I$. For our purposes, we use a constant current $I$ and see how $f$ is affected based on $I$. In order to understand the f-I curve, we must first understand interspike intervals.

An interspike interval (ISI) is the time between two successive spikes. More importantly, though, the ISI is the inverse of the firing rate, as the firing rate is spikes per time, and the ISI is the time per spike.

In order to find the firing rate:

Recall this equation
$$
V_m(t) = V_\infty - (V_\infty - V_L)e^{-\frac{t}{\tau_m}}
$$
The $V_L$ component is actually $V_m(0)$, but we can say this equation models the voltage after a spike because the spike also "resets" the time and voltage.

The next spike occurs at threshold voltage $V_\text{th}$.
$$
V_\text{th} - V_\infty = (V_L - V_\infty)e^\frac{-t}{\tau_m} \\
t = -\tau_m \ln\left({\frac{V_\text{th} - V_\infty}{V_L - V_\infty}}\right)
$$
Because we also have a refractory period, we need to add in that. Thus, our ISI (equal to the inverse of firing rate) is equal to:
$$
t_\text{ref} -\tau_m \ln\left({\frac{V_\text{th} - V_\infty}{V_L - V_\infty}}\right)
$$

However, you may notice that we want to find the f-I curve, not just the firing rate. To do this, we substitute $V_L + RI = V_\infty$ again. Thus, we get
$$
f = \frac{1}{t_\text{ref} -\tau_m \ln\left({\frac{V_\text{th} - V_L - RI}{-RI}}\right)}
$$

One last thing is that we must have a current $I$ such that $V_L + RI > V_\text{th}$, meaning the neuron can fire. Thus, the current must be $I > \frac{V_\text{th} - V_L}{R}$. At smaller currents, no firing occurs. We say $>$ and not $\geq$ because if the two sides are equal, we would only approach the threshold asymptotically, a property of capacitors.

Let's create the f-I curve below, varying current as a `numpy` linspace.

In [ ]:
# zeros_like makes a zero array with the same shape as the given array
currents = np.linspace(0, 0.5e-9, 500)
f = np.zeros_like(currents)

for it, I in enumerate(currents):
    # TODO: fill out the condition for a sufficient current
    if :
        # TODO: calculate the ISI
        #       note: natural log is np.log
        isi =

        # TODO: calculate firing rate
        f[it] =

plt.plot(currents * 1e9, f)
plt.xlabel("Input Current (nA)")
plt.ylabel("Firing Rate (Hz)")
plt.title("f-I Curve")

plt.show()